# SHAP Explainability - Interpreting the Black Box

## Objectives
- Use SHAP (SHapley Additive exPlanations) to interpret the XGBoost model.
- Visualize global feature importance.
- Explain individual predictions (Local Interpretability).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load Data & Model
df = pd.read_csv('../data/processed/final_analytical_dataset.csv')

# Preprocess (Same as XGBoost notebook)
drop_cols = ['customerID', 'Tenure_Cohort', 'Last_Activity_Date', 'AcquisitionDate', 'CohortMonth']
df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])

for col in df_model.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Load Model
with open('../models/model_artifacts/xgb_model.pkl', 'rb') as f:
    model = pickle.load(f)

In [ ]:
# Calculate SHAP Values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

## 1. Global Feature Importance (Summary Plot)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X, plot_type="bar")
plt.show()

## 2. Feature Impact Direction (Beeswarm Plot)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X)
plt.show()

## 3. Local Explainability (Waterfall Plot)

In [ ]:
# Explain the first prediction
shap.plots.waterfall(shap.Explanation(values=shap_values[0], 
                                      base_values=explainer.expected_value, 
                                      data=X.iloc[0], 
                                      feature_names=X.columns))